In [ ]:

"""
OLS (Multiple Linear Regression) with train/test split for hospital admissions prediction.
Follows the same chronological per-city split as QRF.
Computes test-set metrics: RMSE, MAE, R², MAPE.
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -------------------------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------------------------
filename = "merged_project_data.csv"   # Change to your file name
df = pd.read_csv(filename)
print(f"Dataset shape: {df.shape}")

# -------------------------------------------------------------------
# 2. ONE-HOT ENCODE CITY (Delhi = baseline) and REMOVE ORIGINAL CITY COLUMN
# -------------------------------------------------------------------
city_dummies = pd.get_dummies(df['city'], prefix='city', drop_first=True)
df = pd.concat([df.drop('city', axis=1), city_dummies], axis=1)

# -------------------------------------------------------------------
# 3. DEFINE FEATURE COLUMNS AND TARGET
# -------------------------------------------------------------------
feature_cols = [
    'pm25', 'pm10', 'o3', 'no2', 'so2', 'co',
    'pm25_lag1', 'pm25_lag2', 'pm25_lag3',
    'hospital_capacity',
    'city_London', 'city_Mexico City'
]
target = 'hospital_admissions'

# Remove rows with missing values in features or target
df_clean = df[feature_cols + [target]].dropna()
print(f"Rows after cleaning: {len(df_clean)}")

# -------------------------------------------------------------------
# 4. ENSURE ALL FEATURES ARE NUMERIC (convert to float)
# -------------------------------------------------------------------
for col in feature_cols:
    if df_clean[col].dtype == 'object':
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    df_clean[col] = df_clean[col].astype(float)
df_clean = df_clean.dropna()  # just in case new NaNs appeared
print(f"Rows after numeric conversion: {len(df_clean)}")
print("Feature dtypes after conversion:\n", df_clean[feature_cols].dtypes.value_counts())

# -------------------------------------------------------------------
# 5. RECONSTRUCT CITY COLUMN FROM DUMMIES FOR SPLITTING
# -------------------------------------------------------------------
def get_city(row):
    if row['city_London'] == 1:
        return 'London'
    elif row['city_Mexico City'] == 1:
        return 'Mexico City'
    else:
        return 'Delhi'

df_clean['city'] = df_clean.apply(get_city, axis=1)

# -------------------------------------------------------------------
# 6. CHRONOLOGICAL TRAIN/TEST SPLIT PER CITY
# -------------------------------------------------------------------
X_train_list, X_test_list = [], []
y_train_list, y_test_list = [], []

for city in df_clean['city'].unique():
    city_df = df_clean[df_clean['city'] == city].sort_index()  # sort by original index to preserve time order
    X_city = city_df[feature_cols]
    y_city = city_df[target]
    n = len(X_city)
    split_idx = int(0.8 * n)
    X_train_list.append(X_city.iloc[:split_idx])
    X_test_list.append(X_city.iloc[split_idx:])
    y_train_list.append(y_city.iloc[:split_idx])
    y_test_list.append(y_city.iloc[split_idx:])

X_train = pd.concat(X_train_list).sort_index()
X_test = pd.concat(X_test_list).sort_index()
y_train = pd.concat(y_train_list).sort_index()
y_test = pd.concat(y_test_list).sort_index()

# Final check: all numeric
assert all(X_train.dtypes == float), "X_train still has non-float columns"
assert all(X_test.dtypes == float), "X_test still has non-float columns"

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# -------------------------------------------------------------------
# 7. FIT OLS MODEL ON TRAINING SET
# -------------------------------------------------------------------
X_train_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train_const).fit()

print("\n" + "="*80)
print("OLS REGRESSION SUMMARY (TRAINING SET)")
print("="*80)
print(model.summary())

# -------------------------------------------------------------------
# 7b. DISPLAY COEFFICIENTS TABLE (HỆ SỐ HỒI QUY)
# -------------------------------------------------------------------
print("\n" + "="*80)
print("COEFFICIENTS TABLE")
print("="*80)

# Extract coefficient information from the fitted model
coef = model.params
std_err = model.bse
t_values = model.tvalues
p_values = model.pvalues
conf_int = model.conf_int()  # 95% confidence intervals

# Build a clean DataFrame for the coefficients table
coef_table = pd.DataFrame({
    'Coefficient': coef,
    'Std. Error': std_err,
    't-value': t_values,
    'P>|t|': p_values,
    '[0.025': conf_int[0],
    '0.025]': conf_int[1]   # Actually this is the upper bound, but naming preserves convention
})

# Rename the last two columns for clarity
coef_table.rename(columns={'[0.025': '95% CI Lower', '0.025]': '95% CI Upper'}, inplace=True)

# Round to 6 decimal places for readability
coef_table = coef_table.round(6)

# Print the table
print(coef_table.to_string())
print("\nNote: 'const' is the intercept term.")

# -------------------------------------------------------------------
# 8. PREDICT ON TEST SET AND EVALUATE
# -------------------------------------------------------------------
X_test_const = sm.add_constant(X_test)
y_pred = model.predict(X_test_const)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# MAPE (avoid division by zero)
mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100

print("\n" + "="*80)
print("TEST SET PERFORMANCE")
print("="*80)
print(f"RMSE: {rmse:.4f} patients/day")
print(f"MAE : {mae:.4f} patients/day")
print(f"R²  : {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

Dataset shape: (11611, 14)
Rows after cleaning: 11611
Rows after numeric conversion: 11611
Feature dtypes after conversion:
 float64    12
Name: count, dtype: int64
Training samples: 9288
Test samples: 2323

OLS REGRESSION SUMMARY (TRAINING SET)
                             OLS Regression Results                            
Dep. Variable:     hospital_admissions   R-squared:                       0.981
Model:                             OLS   Adj. R-squared:                  0.981
Method:                  Least Squares   F-statistic:                 4.355e+04
Date:                 Mon, 27 Apr 2026   Prob (F-statistic):               0.00
Time:                         09:39:37   Log-Likelihood:                -25981.
No. Observations:                 9288   AIC:                         5.199e+04
Df Residuals:                     9276   BIC:                         5.207e+04
Df Model:                           11                                         
Covariance Type:             nonro